# Chapter 4 Analyzing tabular data with pyspark

This chapter cover:

- Reading delimited data into a PySpark data frame
- Understanding how PySpark represents tabular data in a data frame
- Ingesting and exploring tabular or relational data
- Selecting, manipulating, renaming, and deleting columns in a data frame
- Summarizing data frames for quick exploration

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# change the account name to your email account
account='sli'

# define a root path to access the data in the DataAnalysisWithPythonAndPySpark
data_path='/net/clusterhn/home/'+account+'/isa460/data/'

spark = (SparkSession.builder.appName("Analyzing tabluar data")
        .config("spark.port.maxRetries", "100")
        .getOrCreate())

# confiture the log level (defaulty is WWARN)
spark.sparkContext.setLogLevel('ERROR')

In [ ]:
# import data from a list of lists

my_grocery_list=[
    ["Banana", 2, 1.74],
    ["Apple", 4, 2.04],
    ["Carrot", 1, 1.09],
    ["cake", 1, 10.99]
]

# create a Dataframe based on the list

df=spark.createDataFrame(my_grocery_list, ["Item", "Quantity", "Price"])

df.printSchema()

## import data from a csv file

For this exercise, we’ll use some open data from the government of Canada, more specifically the CRTC (Canadian Radio-Television and Telecommunications Commission). Every broadcaster is mandated to provide a complete log of the programs and commercials showcased to the Canadian public. This gives us a lot of potential questions to answer, but we’ll select just one:
**What are the channels with the greatest and least proportion of commercials?**

You can download the [file](http://mng.bz/y4YJ) on the Canada Open Data portal ; select the BroadcastLogs_2018_Q3_M8 file. The file is 994 MB to download, which might be too large, depending on your computer. The book’s repository contains a sample of the data under the data/broadcast_logs directory, which you can use in place of the original file. You also need to download the Data Dictionary in .doc form, as well as the Reference Tables zip file, unzipping them into a ReferenceTables directory in data/ broadcast_logs. Once again, the examples assume that the data is downloaded under data/broadcast_logs and that PySpark is launched from the root of the repository.

### check the original csv file 

In [ ]:
!head -n 6 {data_path}/broadcast_logs/BroadcastLogs_2018_Q3_M8_sample.CSV

In [ ]:
import os

directory=data_path+'/broadcast_logs/'

logs=spark.read.csv(os.path.join(directory, "BroadcastLogs_2018_Q3_M8_sample.CSV"),
                                 sep="|",
                                 header=True,
                                 inferSchema=True,
                                 timestampFormat="yyyy-MM-dd",)

In [ ]:
logs.printSchema()

## Exploring the shape of our data universe

![Figure 4.4](https://raw.githubusercontent.com/Suhong88/ISA460_Fall2023/main/images/Figure%204.4.png)

## The basics of data manipulation: Selecting, dropping, renaming, ordering, diagnosing

### Select columns

In [ ]:
logs.select("BroadcastLogID", "LogServiceID", "LogDate").show(5, False)

In [ ]:
# four ways of selecting columns
# Using the string to column conversion
logs.select("BroadCastLogID", "LogServiceID", "LogDate")

# use * to unpack a list
logs.select(*["BroadCastLogID", "LogServiceID", "LogDate"])
 
# Passing the column object explicitly
logs.select(
    F.col("BroadCastLogID"), F.col("LogServiceID"), F.col("LogDate")
)
logs.select(
    *[F.col("BroadCastLogID"), F.col("LogServiceID"), F.col("LogDate")]
)

In [ ]:
# for a dataframe with a lot of columns, we can slice the columns into groups to display them by small groups

total_columns=len(logs.columns)

columns_per_row=5

for i in range(0, total_columns, columns_per_row):
    logs.select(logs.columns[_____]).show(5)

### Create DisplayByGroup() function

In [ ]:
# Since we may need to slice columns in the future. We will create a function for later use

def displayByGroup():
    # complete function

In [ ]:
# Use the above function to display 6 columns per row, and 10 records. 

#displayByGroup(logs, 6, 10)

### Store DisplyaByGroup() function into a python file to be used later

### Drop columns

In [ ]:
logs1 = logs.drop("BroadcastLogID", "SequenceNO")

# Testing if we effectively got rid of the columns
 
print("BroadcastLogID" in logs1.columns)  # => False
print("SequenceNO" in logs1.columns)  # => False

In [ ]:
# instead drop, you can also select the ones you want to keep
logs1 = logs.select(
    *[c for c in logs.columns if c not in ["BroadcastLogID", "SequenceNO"]]
)

logs1.columns

### create a column showing duration in seconds

In [ ]:
# check Duration column

logs.select("Duration").show(5)

In [ ]:
# check the data type of Duration

logs.select(F.col("Duration")).dtypes

In [ ]:
# step 1: extract hours, minutes and seconds

logs.select(F.col("Duration").substr(1,2).cast("int").alias("dur_hours"), 
            F.col("Duration").substr(_,_).cast("int").alias("dur_minutes"),
            F.col("Duration").substr(_,_).cast("int").alias("dur_seconds")).show()

In [ ]:
# Step 2: create a new column for duration in seconds

logs.withColumn("Duration_seconds",
                # complete this function)

In [ ]:
# print Schema. Why I do not see the new column?
logs.printSchema()

In [ ]:
# store result in a new dataframe
logs1=logs.withColumn("Duration_seconds",
                # complete this function)

![Warning](https://raw.githubusercontent.com/Suhong88/ISA460_Fall2023/main/images/Figure%204.5.png)

### Rename and Reordering columns

In [ ]:
logs2=logs1.withColumnRenamed("Duration_seconds", "duration_seconds")
logs2.columns

In [ ]:
# change all columns to lower case

logs3=logs2.toDF(*[x.lower() for x in logs.columns])

In [ ]:
# create a function to change all column names to lower caose

def columnLowerCase(df):
   
# add this function our helper functions

In [ ]:
# order all columns in alphabetical order

logs.select(sorted(logs.columns)).printSchema()

# store the result into a new dataframe

#logs1=logs.select(sorted(logs.columns))

### Diagnosing a data frame with describe() and summary()

In [ ]:
logs.describe().show()

In [ ]:
# for a dataframe with a lot of column, we can describe it one by one

for i in logs.columns:
    logs.select(i).describe().show()

### identify numerical columns

In [ ]:
logs.dtypes

In [ ]:
numColumns=[item for item, type in logs.dtypes if type=='int']

numColumns

In [ ]:
# identify categorical columns

catColumns=

In [ ]:
# apply describe to numerical columns
for i in numColumns:
   logs.select(i).describe().show()

## In class exercise

### 1. Create a new data frame, logs_clean, that contains only the columns that do not end with ID.

In [ ]:
import os

directory=data_path+'/broadcast_logs/'

logs=spark.read.csv(os.path.join(directory, "BroadcastLogs_2018_Q3_M8_sample.CSV"),
                                 sep="|",
                                 header=True,
                                 inferSchema=True,
                                 timestampFormat="yyyy-MM-dd",)
logs.columns

### 2. Display a list of program title that includes word apple, remove duplidate.

### 3. Display top 5 program title based on number of times it has been broadcasted.